# Promo email by customer type: CMH test

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Cochran-Mantel-Haenszel Test** · a worked example from the StatNex teaching guide

**The story.** A retailer sent a promo email to some customers and compared purchase rates. But loyalty members were far more likely to get the email, and they buy more anyway. So the analysis is split by customer type.

**The question.** Does the promo email still increase buying once we compare like with like?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/cochran_mantel_haenszel.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Customer type,Promo email,Bought,Did not buy,Total
New customers,Got promo email,9.0,51.0,60.0
,No promo email,14.0,126.0,140.0
Returning customers,Got promo email,30.0,70.0,100.0
,No promo email,18.0,62.0,80.0
Loyalty members,Got promo email,75.0,75.0,150.0
,No promo email,12.0,18.0,30.0
,Total,158.0,402.0,560.0

These cells save `cochran_mantel_haenszel.py`, `freeman_halton.py`, `totals.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile cochran_mantel_haenszel.py
"""Stratified-table statistics that SciPy and statsmodels do not give in the app's form.

cmh_correlation   the CMH correlation statistic, rows and columns scored 1, 2, 3, ...
generalised_cmh   Landis, Heyman and Koch's three statistics for r x c strata
mh_risk_ratio     the Mantel-Haenszel common risk ratio and the SE of its log
logit_pooled      the inverse-variance (Woolf, "logit") pooled ratio and the SE of its log

Used by the Cochran-Mantel-Haenszel test (2 x 2 strata) and the extended Mantel test
(k x 2 strata).
"""

import numpy as np
from scipy import stats


def cmh_correlation(tables):
    """CMH correlation statistic and its P value (chi-square, 1 df).

    Rows and columns are scored 1, 2, 3, ... in table order (SAS PROC FREQ's default, Mantel's equally spaced scores),
    with the hypergeometric variance on n - 1. On one table the statistic is
    (n - 1) r^2, the Mantel-Haenszel chi-square; on 2 x 2 strata it is the textbook CMH
    statistic (statsmodels' test_null_odds(correction=False), R's mantelhaen.test).
    """
    numerator = variance = 0.0
    for table in tables:
        n = table.sum()
        if n < 2:
            continue
        row_totals, col_totals = table.sum(axis=1), table.sum(axis=0)
        x = np.arange(1.0, len(row_totals) + 1)
        y = np.arange(1.0, len(col_totals) + 1)
        observed = (np.outer(x, y) * table).sum()
        expected = (x * row_totals).sum() * (y * col_totals).sum() / n
        var_x = (x**2 * row_totals).sum() - (x * row_totals).sum() ** 2 / n
        var_y = (y**2 * col_totals).sum() - (y * col_totals).sum() ** 2 / n
        numerator += observed - expected
        variance += var_x * var_y / (n - 1)
    statistic = numerator**2 / variance
    return statistic, stats.chi2.sf(statistic, 1)


def mh_risk_ratio(tables):
    """Mantel-Haenszel common risk ratio of column 1 (row 1 over row 2), and SE(log).

    The variance is Greenland & Robins (1985). statsmodels' StratifiedTable has the
    estimate (riskratio_pooled) but no interval.
    """
    a, b, c, d = (np.array([t[i, j] for t in tables], float)
                  for i, j in ((0, 0), (0, 1), (1, 0), (1, 1)))
    n = a + b + c + d
    row1, row2 = a + b, c + d
    numerator, denominator = (a * row2 / n).sum(), (c * row1 / n).sum()
    estimate = numerator / denominator if denominator > 0 else np.nan
    if numerator > 0 and denominator > 0:
        variance = ((row1 * row2 * (a + c) - a * c * n) / n**2).sum() / (numerator * denominator)
        return estimate, np.sqrt(variance)
    return estimate, np.nan


def logit_pooled(estimates, variances):
    """Inverse-variance pooled ratio exp(sum w log r / sum w), w = 1 / Var(log r), and SE(log).

    A stratum with an empty cell has no finite log ratio or variance, so it is left out;
    with none left the estimator is undefined -- which is why the Mantel-Haenszel form,
    a ratio of sums, is reported beside it.
    """
    usable = (np.isfinite(estimates) & (estimates > 0)
              & np.isfinite(variances) & (variances > 0))
    if not usable.any():
        return np.nan, np.nan
    weights = 1 / variances[usable]
    estimate = np.exp((weights * np.log(estimates[usable])).sum() / weights.sum())
    return estimate, 1 / np.sqrt(weights.sum())


def generalised_cmh(tables):
    """[(name, df, statistic, P)] for strata larger than 2 x 2 (Landis, Heyman & Koch 1978).

    Each statistic is G' V^-1 G, with G the sum over strata of B (n - m) -- observed
    minus expected counts under conditional independence, the table stacked column by
    column -- and V the sum of B Cov B', Cov = N^2/(N-1) (V_col (x) V_row). They differ
    only in B: every cell but the last row and column (general association), each
    row's sum of column scores (row mean scores), or the score product (correlation).
    The scores are cmh_correlation's table scores, so the last statistic is
    that one.
    """
    rows, cols = tables[0].shape
    keep_rows, keep_cols = np.eye(rows)[:-1], np.eye(cols)[:-1]
    names = ("General association", "Row mean scores differ", "Correlation")
    sums, variances = [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]
    for table in tables:
        n = table.sum()
        if n < 2:
            continue
        row_totals, col_totals = table.sum(axis=1), table.sum(axis=0)
        p_row, p_col = row_totals / n, col_totals / n
        covariance = n**2 / (n - 1) * np.kron(np.diag(p_col) - np.outer(p_col, p_col),
                                              np.diag(p_row) - np.outer(p_row, p_row))
        gap = (table - np.outer(row_totals, col_totals) / n).flatten(order="F")
        x = np.arange(1.0, len(row_totals) + 1)
        y = np.arange(1.0, len(col_totals) + 1)
        for k, contrast in enumerate((np.kron(keep_cols, keep_rows),
                                      np.kron(y[None, :], keep_rows),
                                      np.kron(y[None, :], x[None, :]))):
            sums[k] = sums[k] + contrast @ gap
            variances[k] = variances[k] + contrast @ covariance @ contrast.T
    out = []
    for name, total, variance in zip(names, sums, variances):
        variance = np.atleast_2d(variance)
        df = np.linalg.matrix_rank(variance)
        statistic = np.atleast_1d(total) @ np.linalg.pinv(variance) @ np.atleast_1d(total)
        out.append((name, df, statistic, stats.chi2.sf(statistic, df)))
    return out

In [ ]:
%%writefile freeman_halton.py
"""Fisher's exact test for a table larger than 2x2 (Freeman and Halton 1951).

scipy.stats.fisher_exact estimates the P value of an r x c table by Monte Carlo; this
computes it exactly, as R's fisher.test does. With both margins fixed, a table's
probability is prod(r_i!) prod(c_j!) / (n! prod(n_ij!)), and P sums it over every
table no more probable than the observed one (within a relative 1e-7, R's tolerance,
so tables tied in exact arithmetic are not split by rounding).
"""

import math
from itertools import combinations

import numpy as np
from scipy import special, stats

#: Past this many tables the P value is a Monte Carlo estimate, as in the app.
EXACT_TABLE_LIMIT = 2_000_000
MONTE_CARLO_TABLES = 200_000
MONTE_CARLO_SEED = 20240501


def compositions(total, parts):
    """Every way to split ``total`` into ``parts`` non-negative counts, one per row."""
    if parts == 1:
        return np.array([[total]])
    bars = np.array(list(combinations(range(total + parts - 1), parts - 1)))
    edges = np.column_stack([np.full(len(bars), -1), bars, np.full(len(bars), total + parts - 1)])
    return np.diff(edges, axis=1) - 1


def exact_p(counts):
    """The exact P value, or None when there are too many tables to list.

    The tables are built a column at a time: each partial table is a set of row totals
    still to fill and the log of its prod(n_ij!) so far.
    """
    counts = np.asarray(counts, dtype=np.int64)
    if counts.shape[0] > counts.shape[1]:
        counts = counts.T
    rows, cols = counts.sum(axis=1), counts.sum(axis=0)
    n = int(counts.sum())
    log_factorial = special.gammaln(np.arange(n + 1) + 1.0)
    observed = log_factorial[counts].sum()
    remaining = rows[None, :].copy()
    partial = np.zeros(1)
    for total in cols[:-1]:
        if math.comb(int(total) + len(rows) - 1, len(rows) - 1) > EXACT_TABLE_LIMIT:
            return None
        splits = compositions(int(total), len(rows))
        fits = (splits[None, :, :] <= remaining[:, None, :]).all(axis=2)
        state, split = np.nonzero(fits)
        if len(state) > EXACT_TABLE_LIMIT:
            return None
        remaining = remaining[state] - splits[split]
        partial = partial[state] + log_factorial[splits[split]].sum(axis=1)
    weight = partial + log_factorial[remaining].sum(axis=1)
    constant = log_factorial[rows].sum() + log_factorial[cols].sum() - log_factorial[n]
    return float(np.exp(constant - weight[weight >= observed - 1e-7]).sum())


def freeman_halton(counts):
    """(P value, how it was found): exact, or Monte Carlo from a fixed seed."""
    p = exact_p(counts)
    if p is not None:
        return p, "exact"
    result = stats.fisher_exact(
        np.asarray(counts, dtype=np.int64),
        method=stats.MonteCarloMethod(n_resamples=MONTE_CARLO_TABLES,
                                      rng=np.random.default_rng(MONTE_CARLO_SEED)))
    return float(result.pvalue), f"Monte Carlo, {MONTE_CARLO_TABLES} tables"

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import StratifiedTable, Table2x2
from cochran_mantel_haenszel import cmh_correlation, generalised_cmh, logit_pooled, mh_risk_ratio
from freeman_halton import freeman_halton

pd.set_option("display.width", 120)

## The data

In [ ]:
# One table per stratum, stacked: usually 2 x 2, the rows of a stratum exposed / not
# exposed and the columns cases / controls; every stratum has the same shape.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# A stratum's name is printed on its first row only; carry it down to the rows below.
stratum = data["Customer type"].ffill()
labels = data["Promo email"]
# Rows and columns named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) are margins, not data.
# The app always leaves them out of a stratified table, ticked or not. Spaces,
# underscores, case and full-width letters are ignored when names are compared.
from totals import is_total

columns = [name for name in ["Bought", "Did not buy"] if not is_total(name)]
keep = [not is_total(label) for label in labels]
kept = data.loc[keep, columns].astype(float).set_axis(labels[keep])
strata = {name: table for name, table in kept.groupby(stratum[keep].to_numpy(), sort=False)}
# A stratum's rows are matched to the other strata's by their place, and named from the
# first stratum's, as the app names them.
row_names = next(iter(strata.values())).index
strata = {name: table.set_axis(row_names) for name, table in strata.items()}
tables = [table.to_numpy() for table in strata.values()]
two_by_two = tables[0].shape == (2, 2)

## Observed frequency

In [ ]:
for name, table in strata.items():
    print(f"\n{name}")
    print(table)

## Test of independence within each stratum

In [ ]:
# Each stratum on its own. The Mantel-Haenszel chi-square is the correlation statistic
# on one table, (n - 1) r^2, on 1 degree of freedom.
stratum_tests = []
for name, table in strata.items():
    counts = table.to_numpy()
    pearson = stats.chi2_contingency(counts, correction=False)
    likelihood = stats.chi2_contingency(counts, correction=False, lambda_="log-likelihood")
    mantel_haenszel, mantel_haenszel_p = cmh_correlation([counts])
    # Larger than 2 x 2, Fisher's test is the Freeman-Halton extension.
    fisher = stats.fisher_exact(counts).pvalue if two_by_two else freeman_halton(counts)[0]
    cramers_v = np.sqrt(pearson.statistic / (counts.sum() * (min(counts.shape) - 1)))
    tests = pd.DataFrame(
        [["Pearson chi-square", pearson.dof, pearson.statistic, pearson.pvalue],
         ["Likelihood-ratio chi-square", likelihood.dof, likelihood.statistic, likelihood.pvalue],
         ["Mantel-Haenszel chi-square", 1, mantel_haenszel, mantel_haenszel_p],
         ["Fisher's exact test", np.nan, np.nan, fisher],
         ["Cramer's V", np.nan, cramers_v, np.nan]],
        columns=["test", "df", "value", "P"])
    stratum_tests.append(tests)
    print(f"\n{name}")
    print(tests.to_string(index=False))

# The ratios, their homogeneity and the pooled estimates are 2 x 2 quantities; strata
# larger than 2 x 2 get the generalised CMH statistics instead.
if two_by_two:
    # === Risk ratio and odds ratio within each stratum ===
    # Intervals on the log scale, as statsmodels' Table2x2 builds them. The column-2 risk
    # ratio is the same calculation with the two columns swapped. A stratum with an empty
    # cell has no ratio, and the app reports none rather than add 0.5 to every cell.
    level = 95 / 100
    z = stats.norm.ppf(0.5 + level / 2)
    stratum_ratios = []
    for name, table in strata.items():
        counts = table.to_numpy()
        if (counts == 0).any():
            values = [[np.nan, np.nan, np.nan]] * 3
        else:
            fourfold = Table2x2(counts, shift_zeros=False)
            swapped = Table2x2(counts[:, ::-1], shift_zeros=False)
            values = [[fourfold.oddsratio, *fourfold.oddsratio_confint(alpha=1 - level)],
                      [fourfold.riskratio, *fourfold.riskratio_confint(alpha=1 - level)],
                      [swapped.riskratio, *swapped.riskratio_confint(alpha=1 - level)]]
        ratios = pd.DataFrame(values, columns=["value", "lower", "upper"],
                              index=["Case-control (odds ratio)", "Cohort (risk ratio, column 1)",
                                     "Cohort (risk ratio, column 2)"])
        stratum_ratios.append(ratios)
        print(f"\n{name}: {level:.0%} confidence interval")
        print(ratios)

    # === Homogeneity of the odds ratio (Breslow-Day) ===
    # Whether the strata share one odds ratio. A small P means the pooled odds ratio below
    # averages things that are not the same. adjust=False: Breslow and Day's original form,
    # without Tarone's correction.
    stratified = StratifiedTable(tables)
    breslow_day = stratified.test_equal_odds(adjust=False)
    homogeneity = pd.DataFrame(
        [["Breslow-Day", len(tables) - 1, breslow_day.statistic, breslow_day.pvalue]],
        columns=["test", "df", "value", "P"])
    print()
    print(homogeneity.to_string(index=False))

    # === Conditional independence (CMH) ===
    # Differences are added across the strata first and squared after, so strata that lean
    # the same way add up. The rows and columns are scored 1, 2, 3, ... in table order (SAS's
    # default), so on 2 x 2 strata this is the textbook Mantel-Haenszel chi-square, which
    # statsmodels' stratified.test_null_odds(correction=False) also gives.
    cmh, cmh_p = cmh_correlation(tables)
    conditional = pd.DataFrame([["CMH correlation statistic", 1, cmh, cmh_p]],
                               columns=["test", "df", "value", "P"])
    print()
    print(conditional.to_string(index=False))

    # === Adjusted (pooled) risk ratio and odds ratio ===
    # Two estimators for each. Mantel-Haenszel is a ratio of sums, so it survives an empty
    # cell; the logit estimator averages log ratios weighted by 1 / variance, and leaves out
    # any stratum with an empty cell. statsmodels gives the Mantel-Haenszel odds ratio with
    # its Robins-Breslow-Greenland interval; the rest is in cochran_mantel_haenszel.py.
    a, b, c, d = (np.array([t[i, j] for t in tables]) for i, j in ((0, 0), (0, 1), (1, 0), (1, 1)))


    def interval(estimate, log_se):
        return [estimate, estimate * np.exp(-z * log_se), estimate * np.exp(z * log_se)]


    with np.errstate(divide="ignore", invalid="ignore"):
        odds_logit = logit_pooled(a * d / (b * c), 1 / a + 1 / b + 1 / c + 1 / d)
        risk1_logit = logit_pooled((a / (a + b)) / (c / (c + d)),
                                   1 / a - 1 / (a + b) + 1 / c - 1 / (c + d))
        risk2_logit = logit_pooled((b / (a + b)) / (d / (c + d)),
                                   1 / b - 1 / (a + b) + 1 / d - 1 / (c + d))
    pooled = pd.DataFrame(
        [["Common odds ratio", "Mantel-Haenszel", stratified.oddsratio_pooled,
          *stratified.oddsratio_pooled_confint(alpha=1 - level)],
         ["Common odds ratio", "logit", *interval(*odds_logit)],
         ["Common risk ratio, column 1", "Mantel-Haenszel", *interval(*mh_risk_ratio(tables))],
         ["Common risk ratio, column 1", "logit", *interval(*risk1_logit)],
         ["Common risk ratio, column 2", "Mantel-Haenszel",
          *interval(*mh_risk_ratio([t[:, ::-1] for t in tables]))],
         ["Common risk ratio, column 2", "logit", *interval(*risk2_logit)]],
        columns=["measure", "estimator", "value", "lower", "upper"])
    print(f"\n{level:.0%} confidence interval")
    print(pooled.to_string(index=False))
else:
    # === Conditional independence (generalised CMH) ===
    # Landis, Heyman and Koch's three statistics: general association on (r-1)(c-1) df,
    # row mean scores on r-1, and correlation on 1, the last being the CMH correlation
    # statistic of a 2 x 2 analysis.
    conditional = pd.DataFrame(generalised_cmh(tables), columns=["test", "df", "value", "P"])
    print()
    print(conditional.to_string(index=False))

## What the result says

The Cochran-Mantel-Haenszel (CMH) test compares like with like: it looks at the email's effect inside each customer type and then combines the results. The odds ratio says how many times higher the odds of buying are with the email. It is 1.59 for new customers, 1.48 for returning customers and 1.50 for loyalty members. The Breslow-Day test (P = 0.99) gives no sign that these three differ, so one common value is fair.

The common odds ratio is 1.51, but its confidence interval, the range of plausible values, runs from 0.96 to 2.36 and includes 1, which means no effect. The CMH statistic is 3.26 with P = 0.071, above 0.05.

So an effect of the email is not clearly shown. This does not prove the email does nothing.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.